In [1]:
!pip install gradio reportlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 18.1 MB/s eta 0:00:00


In [2]:
%%writefile analyzer.py
"""analyzer.py - AST analysis: extracts all function/class metadata."""
import ast
from typing import Optional

MATH_OPS = {ast.Add, ast.Sub, ast.Mult, ast.Div, ast.Mod, ast.Pow, ast.FloorDiv}
DB_KEYWORDS = {"execute","commit","rollback","fetchall","fetchone","query","insert",
    "update","delete","select","cursor","session","filter","add","merge","flush","connection"}
FILE_KEYWORDS = {"open","read","write","close","readline","readlines","seek","flush"}
API_KEYWORDS = {"requests","get","post","put","patch","delete","fetch","urlopen",
    "http","https","response","json","aiohttp","httpx","urllib"}
VALIDATION_KEYWORDS = {"validate","check","verify","assert","is_valid","ensure","raise",
    "valueerror","typeerror","exception"}
TRANSFORM_KEYWORDS = {"map","filter","reduce","zip","enumerate","sorted","list","dict",
    "set","tuple","join","split","strip","replace","format","encode","decode","convert",
    "parse","serialize","deserialize","transform","process"}


def _node_to_str(node) -> Optional[str]:
    if node is None:
        return None
    try:
        return ast.unparse(node)
    except Exception:
        return None


class FunctionMetadata:
    def __init__(self):
        self.name = ""
        self.is_async = False
        self.is_method = False
        self.class_name = None
        self.lineno = 0
        self.end_lineno = 0
        self.decorators = []
        self.params = []
        self.return_annotation = None
        self.raised_exceptions = []
        self.has_return = False
        self.return_values = []
        self.existing_docstring = None
        self.inferred_logic = []
        self.has_yield = False
        self.has_side_effects = False
        self.complexity = 0
        self.complexity_label = "Low"

    def to_dict(self):
        return self.__dict__.copy()


class ClassMetadata:
    def __init__(self):
        self.name = ""
        self.lineno = 0
        self.bases = []
        self.decorators = []
        self.existing_docstring = None
        self.methods = []


class LogicInferencer(ast.NodeVisitor):
    def __init__(self):
        self.logic_tags = set()
        self.raised = set()
        self.has_return = False
        self.return_values = []
        self.has_yield = False
        self.has_side_effects = False

    def visit_BinOp(self, node):
        if type(node.op) in MATH_OPS:
            self.logic_tags.add("Performs mathematical calculations")
        self.generic_visit(node)

    def visit_AugAssign(self, node):
        if type(node.op) in MATH_OPS:
            self.logic_tags.add("Performs mathematical calculations")
        self.generic_visit(node)

    def visit_Call(self, node):
        func_name = ""
        if isinstance(node.func, ast.Name):
            func_name = node.func.id
        elif isinstance(node.func, ast.Attribute):
            func_name = node.func.attr
        lower = func_name.lower()
        if func_name in FILE_KEYWORDS or lower in FILE_KEYWORDS:
            self.logic_tags.add("Performs file I/O operations")
            self.has_side_effects = True
        if lower in DB_KEYWORDS:
            self.logic_tags.add("Executes database / CRUD operations")
            self.has_side_effects = True
        if lower in API_KEYWORDS:
            self.logic_tags.add("Makes API or network calls")
            self.has_side_effects = True
        if lower in VALIDATION_KEYWORDS:
            self.logic_tags.add("Performs input validation or data checks")
        if lower in TRANSFORM_KEYWORDS:
            self.logic_tags.add("Transforms or processes data structures")
        if lower in {"print","logging","logger","log","warn","error","info","debug"}:
            self.has_side_effects = True
        self.generic_visit(node)

    def visit_For(self, node):
        self.logic_tags.add("Iterates over a sequence or collection")
        self.generic_visit(node)

    def visit_While(self, node):
        self.logic_tags.add("Contains loop-based iteration logic")
        self.generic_visit(node)

    def visit_Return(self, node):
        self.has_return = True
        if node.value is not None:
            v = _node_to_str(node.value)
            if v:
                self.return_values.append(v)
        self.generic_visit(node)

    def visit_Yield(self, node):
        self.has_yield = True
        self.generic_visit(node)

    def visit_YieldFrom(self, node):
        self.has_yield = True
        self.generic_visit(node)

    def visit_Raise(self, node):
        if node.exc is not None:
            exc_str = _node_to_str(node.exc) or ""
            exc_name = exc_str.split("(")[0].strip()
            if exc_name:
                self.raised.add(exc_name)
        self.generic_visit(node)

    def visit_With(self, node):
        for item in node.items:
            ctx = _node_to_str(item.context_expr) or ""
            if "open" in ctx:
                self.logic_tags.add("Performs file I/O operations")
                self.has_side_effects = True
        self.generic_visit(node)

    def visit_AsyncWith(self, node):
        self.logic_tags.add("Uses async context manager (likely I/O or network)")
        self.has_side_effects = True
        self.generic_visit(node)

    def visit_Await(self, node):
        self.logic_tags.add("Uses async/await for non-blocking operations")
        self.generic_visit(node)


class CodeAnalyzer(ast.NodeVisitor):
    def __init__(self, source_code):
        self.source_code = source_code
        self.functions = []
        self.classes = []
        self._current_class = None
        self._tree = None
        self._visited = set()

    def analyze(self):
        try:
            self._tree = ast.parse(self.source_code)
        except SyntaxError as e:
            return {"error": f"Syntax error: {e}"}
        self.visit(self._tree)
        self._compute_complexities()
        total = len(self.functions)
        documented = sum(1 for f in self.functions if f.existing_docstring)
        coverage = round((documented / total) * 100, 1) if total else 100.0
        return {
            "functions": [f.to_dict() for f in self.functions],
            "classes": [{"name": c.name, "lineno": c.lineno, "bases": c.bases,
                         "decorators": c.decorators,
                         "existing_docstring": c.existing_docstring,
                         "method_count": len(c.methods)} for c in self.classes],
            "total_functions": total,
            "documented": documented,
            "coverage": coverage,
        }

    def _extract_params(self, args):
        params = []
        all_args = args.posonlyargs + args.args
        offset = len(all_args) - len(args.defaults)
        for i, arg in enumerate(all_args):
            di = i - offset
            default = _node_to_str(args.defaults[di]) if di >= 0 else None
            params.append({"name": arg.arg,
                           "annotation": _node_to_str(arg.annotation),
                           "default": default})
        if args.vararg:
            params.append({"name": f"*{args.vararg.arg}",
                           "annotation": _node_to_str(args.vararg.annotation),
                           "default": None})
        for i, arg in enumerate(args.kwonlyargs):
            kd = args.kw_defaults
            default = _node_to_str(kd[i]) if i < len(kd) and kd[i] else None
            params.append({"name": arg.arg,
                           "annotation": _node_to_str(arg.annotation),
                           "default": default})
        if args.kwarg:
            params.append({"name": f"**{args.kwarg.arg}",
                           "annotation": _node_to_str(args.kwarg.annotation),
                           "default": None})
        return params

    def _get_docstring(self, node):
        if (node.body and isinstance(node.body[0], ast.Expr)
                and isinstance(node.body[0].value, ast.Constant)
                and isinstance(node.body[0].value.value, str)):
            return node.body[0].value.value
        return None

    def _process_func(self, node):
        key = (node.name, node.lineno)
        if key in self._visited:
            return
        self._visited.add(key)
        meta = FunctionMetadata()
        meta.name = node.name
        meta.is_async = isinstance(node, ast.AsyncFunctionDef)
        meta.lineno = node.lineno
        meta.end_lineno = getattr(node, "end_lineno", node.lineno)
        meta.decorators = [_node_to_str(d) or "<dec>" for d in node.decorator_list]
        meta.params = self._extract_params(node.args)
        meta.return_annotation = _node_to_str(node.returns)
        meta.existing_docstring = self._get_docstring(node)
        if self._current_class:
            meta.is_method = True
            meta.class_name = self._current_class.name
            if meta.params and meta.params[0]["name"] in ("self", "cls"):
                meta.params = meta.params[1:]
        inf = LogicInferencer()
        for child in node.body:
            inf.visit(child)
        meta.inferred_logic = sorted(inf.logic_tags)
        meta.raised_exceptions = sorted(inf.raised)
        meta.has_return = inf.has_return
        meta.return_values = inf.return_values[:3]
        meta.has_yield = inf.has_yield
        meta.has_side_effects = inf.has_side_effects
        self.functions.append(meta)
        if self._current_class:
            self._current_class.methods.append(meta)

    def visit_FunctionDef(self, node):
        self._process_func(node)
        for child in ast.walk(node):
            if child is not node and isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                self._process_func(child)

    def visit_AsyncFunctionDef(self, node):
        self._process_func(node)
        for child in ast.walk(node):
            if child is not node and isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                self._process_func(child)

    def visit_ClassDef(self, node):
        cls = ClassMetadata()
        cls.name = node.name
        cls.lineno = node.lineno
        cls.bases = [_node_to_str(b) or "" for b in node.bases]
        cls.decorators = [_node_to_str(d) or "<dec>" for d in node.decorator_list]
        cls.existing_docstring = self._get_docstring(node)
        prev = self._current_class
        self._current_class = cls
        self.classes.append(cls)
        self.generic_visit(node)
        self._current_class = prev

    def _compute_complexities(self):
        from complexity import compute_complexity, classify_complexity
        if not self._tree:
            return
        for fm in self.functions:
            for node in ast.walk(self._tree):
                if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    if node.name == fm.name and node.lineno == fm.lineno:
                        fm.complexity = compute_complexity(node)
                        fm.complexity_label = classify_complexity(fm.complexity)
                        break


def analyze_source(source_code):
    analyzer = CodeAnalyzer(source_code)
    result = analyzer.analyze()
    return analyzer, result

def analyze_file(filepath):
    with open(filepath, "r", encoding="utf-8") as f:
        source = f.read()
    return CodeAnalyzer(source).analyze()

Writing analyzer.py


In [3]:
%%writefile generator.py
# paste full content of generator.py here
"""generator.py - Docstring generation in Google, NumPy, and Sphinx styles."""
import textwrap
from typing import Optional


def _type_str(annotation: Optional[str], default: Optional[str] = None) -> str:
    if annotation:
        return annotation
    if default is not None:
        try:
            eval(default)
            t = type(eval(default)).__name__
            return t
        except Exception:
            pass
    return "Any"


def _build_summary(meta: dict) -> str:
    """Craft a one-line summary based on inferred logic and function name."""
    name = meta["name"]
    logic = meta.get("inferred_logic", [])
    is_async = meta.get("is_async", False)
    async_prefix = "Asynchronously " if is_async else ""

    # Priority-ordered summary hints
    if "Makes API or network calls" in logic:
        return f"{async_prefix}Sends a network request and returns the response for `{name}`."
    if "Executes database / CRUD operations" in logic:
        return f"{async_prefix}Performs a database operation via `{name}`."
    if "Performs file I/O operations" in logic:
        return f"{async_prefix}Handles file I/O operations in `{name}`."
    if "Performs mathematical calculations" in logic:
        return f"Computes a mathematical result using `{name}`."
    if "Performs input validation or data checks" in logic:
        return f"Validates input data and enforces constraints in `{name}`."
    if "Transforms or processes data structures" in logic:
        return f"Transforms or processes data structures in `{name}`."
    if "Iterates over a sequence or collection" in logic:
        return f"Iterates over a collection and processes each element in `{name}`."
    if meta.get("has_yield"):
        return f"Generator function that yields values from `{name}`."

    # Fallback: humanise the function name
    words = name.replace("_", " ").strip()
    return f"{async_prefix}{words.capitalize()}."


def _notes_section(meta: dict) -> list[str]:
    notes = []
    if meta.get("has_side_effects"):
        notes.append("This function produces side effects (I/O, network, or logging).")
    if meta.get("is_async"):
        notes.append("This is a coroutine; use `await` when calling.")
    if meta.get("has_yield"):
        notes.append("This is a generator function.")
    if meta.get("complexity_label") in ("Medium", "High"):
        notes.append(f"Cyclomatic complexity: {meta.get('complexity_label')} ({meta.get('complexity')}).")
    return notes


def generate_google_docstring(meta: dict, indent: int = 4) -> str:
    pad = " " * indent
    lines = [f'"""{_build_summary(meta)}']

    logic = meta.get("inferred_logic", [])
    if logic:
        lines.append("")
        for tag in logic:
            lines.append(f"{pad}{tag}.")

    params = [p for p in meta.get("params", []) if not p["name"].startswith("*")]
    if params:
        lines.append("")
        lines.append(f"{pad}Args:")
        for p in params:
            type_hint = _type_str(p.get("annotation"), p.get("default"))
            default_note = f" Defaults to {p['default']}." if p.get("default") else ""
            lines.append(f"{pad}    {p['name']} ({type_hint}):{default_note} Description of {p['name']}.")

    ret = meta.get("return_annotation")
    has_return = meta.get("has_return") or meta.get("has_yield")
    if has_return or ret:
        lines.append("")
        lines.append(f"{pad}Returns:")
        ret_type = ret or "Any"
        lines.append(f"{pad}    {ret_type}: Description of the return value.")

    exc = meta.get("raised_exceptions", [])
    if exc:
        lines.append("")
        lines.append(f"{pad}Raises:")
        for e in exc:
            lines.append(f"{pad}    {e}: When an error condition is encountered.")

    notes = _notes_section(meta)
    if notes:
        lines.append("")
        lines.append(f"{pad}Note:")
        for n in notes:
            lines.append(f"{pad}    {n}")

    lines.append(f'{pad}"""')
    return "\n".join(lines)


def generate_numpy_docstring(meta: dict, indent: int = 4) -> str:
    pad = " " * indent
    sep = "-" * 10
    lines = [f'"""{_build_summary(meta)}']

    logic = meta.get("inferred_logic", [])
    if logic:
        lines.append("")
        lines.append(f"{pad}Extended Summary")
        lines.append(f"{pad}{sep}")
        for tag in logic:
            lines.append(f"{pad}{tag}.")

    params = [p for p in meta.get("params", []) if not p["name"].startswith("*")]
    if params:
        lines.append("")
        lines.append(f"{pad}Parameters")
        lines.append(f"{pad}{sep}")
        for p in params:
            type_hint = _type_str(p.get("annotation"), p.get("default"))
            lines.append(f"{pad}{p['name']} : {type_hint}")
            default_note = f"Default is {p['default']}. " if p.get("default") else ""
            lines.append(f"{pad}    {default_note}Description of {p['name']}.")

    ret = meta.get("return_annotation")
    has_return = meta.get("has_return") or meta.get("has_yield")
    if has_return or ret:
        lines.append("")
        lines.append(f"{pad}Returns")
        lines.append(f"{pad}{sep}")
        ret_type = ret or "Any"
        lines.append(f"{pad}result : {ret_type}")
        lines.append(f"{pad}    Description of the return value.")

    exc = meta.get("raised_exceptions", [])
    if exc:
        lines.append("")
        lines.append(f"{pad}Raises")
        lines.append(f"{pad}{sep}")
        for e in exc:
            lines.append(f"{pad}{e}")
            lines.append(f"{pad}    When an error condition is encountered.")

    notes = _notes_section(meta)
    if notes:
        lines.append("")
        lines.append(f"{pad}Notes")
        lines.append(f"{pad}{sep}")
        for n in notes:
            lines.append(f"{pad}{n}")

    lines.append(f'{pad}"""')
    return "\n".join(lines)


def generate_sphinx_docstring(meta: dict, indent: int = 4) -> str:
    pad = " " * indent
    lines = [f'"""{_build_summary(meta)}']

    logic = meta.get("inferred_logic", [])
    if logic:
        lines.append("")
        for tag in logic:
            lines.append(f"{pad}{tag}.")

    params = [p for p in meta.get("params", []) if not p["name"].startswith("*")]
    if params:
        lines.append("")
        for p in params:
            type_hint = _type_str(p.get("annotation"), p.get("default"))
            default_note = f" (default: {p['default']})" if p.get("default") else ""
            lines.append(f"{pad}:param {p['name']}: Description of {p['name']}{default_note}.")
            lines.append(f"{pad}:type {p['name']}: {type_hint}")

    ret = meta.get("return_annotation")
    has_return = meta.get("has_return") or meta.get("has_yield")
    if has_return or ret:
        lines.append("")
        ret_type = ret or "Any"
        lines.append(f"{pad}:returns: Description of the return value.")
        lines.append(f"{pad}:rtype: {ret_type}")

    exc = meta.get("raised_exceptions", [])
    if exc:
        lines.append("")
        for e in exc:
            lines.append(f"{pad}:raises {e}: When an error condition is encountered.")

    notes = _notes_section(meta)
    if notes:
        lines.append("")
        lines.append(f"{pad}.. note::")
        for n in notes:
            lines.append(f"{pad}    {n}")

    lines.append(f'{pad}"""')
    return "\n".join(lines)


def generate_docstring(meta: dict, style: str = "google", indent: int = 4) -> str:
    """Generate a docstring in the specified style."""
    style = style.lower()
    if style == "numpy":
        return generate_numpy_docstring(meta, indent)
    elif style in ("sphinx", "rst", "restructuredtext"):
        return generate_sphinx_docstring(meta, indent)
    return generate_google_docstring(meta, indent)


def improve_docstring(existing: str, meta: dict, style: str = "google", indent: int = 4) -> str:
    """
    Improve an existing docstring by merging preserved content with newly
    generated sections for missing parts (Args, Returns, Raises, Notes).
    """
    new_doc = generate_docstring(meta, style, indent)
    # If existing is very short (likely auto-placeholder), replace
    if len(existing.strip()) < 20:
        return new_doc

    # Otherwise, keep existing summary but append missing sections
    existing_stripped = existing.strip()
    pad = " " * indent

    # Check what sections are already present
    has_args = any(kw in existing_stripped for kw in ["Args:", "Parameters", ":param"])
    has_returns = any(kw in existing_stripped for kw in ["Returns:", "Returns\n", ":returns"])
    has_raises = any(kw in existing_stripped for kw in ["Raises:", "Raises\n", ":raises"])

    additions = []
    style_l = style.lower()

    params = [p for p in meta.get("params", []) if not p["name"].startswith("*")]
    if not has_args and params:
        if style_l == "numpy":
            additions.append(f"\n{pad}Parameters\n{pad}----------")
            for p in params:
                t = _type_str(p.get("annotation"), p.get("default"))
                additions.append(f"{pad}{p['name']} : {t}\n{pad}    Description of {p['name']}.")
        elif style_l == "sphinx":
            for p in params:
                t = _type_str(p.get("annotation"), p.get("default"))
                additions.append(f"\n{pad}:param {p['name']}: Description.\n{pad}:type {p['name']}: {t}")
        else:
            additions.append(f"\n{pad}Args:")
            for p in params:
                t = _type_str(p.get("annotation"), p.get("default"))
                additions.append(f"{pad}    {p['name']} ({t}): Description of {p['name']}.")

    exc = meta.get("raised_exceptions", [])
    if not has_raises and exc:
        if style_l == "numpy":
            additions.append(f"\n{pad}Raises\n{pad}------")
            for e in exc:
                additions.append(f"{pad}{e}\n{pad}    When an error condition occurs.")
        elif style_l == "sphinx":
            for e in exc:
                additions.append(f"\n{pad}:raises {e}: When an error condition occurs.")
        else:
            additions.append(f"\n{pad}Raises:")
            for e in exc:
                additions.append(f"{pad}    {e}: When an error condition occurs.")

    if not additions:
        return f'"""{existing_stripped}\n{pad}"""'

    body = f'"""{existing_stripped}\n' + "\n".join(additions) + f'\n{pad}"""'
    return body

Writing generator.py


In [4]:
%%writefile complexity.py
# paste full content of complexity.py here
"""complexity.py - Cyclomatic complexity calculator for Python AST nodes."""
import ast

BRANCH_NODES = (
    ast.If, ast.For, ast.While, ast.ExceptHandler,
    ast.With, ast.AsyncWith, ast.AsyncFor,
    ast.comprehension,
)

def compute_complexity(func_node) -> int:
    """Count branch points to estimate cyclomatic complexity."""
    score = 1  # base complexity
    for node in ast.walk(func_node):
        if isinstance(node, BRANCH_NODES):
            score += 1
        elif isinstance(node, ast.BoolOp):
            score += len(node.values) - 1
        elif isinstance(node, (ast.Try,)):
            score += len(node.handlers)
    return score

def classify_complexity(score: int) -> str:
    if score <= 5:
        return "Low"
    elif score <= 10:
        return "Medium"
    return "High"

Writing complexity.py


In [5]:
%%writefile pdf_exporter.py
# paste full content here
"""pdf_exporter.py - Generate a professional PDF documentation summary using ReportLab."""
import os
from datetime import datetime
from typing import Optional

try:
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib.units import cm
    from reportlab.lib import colors
    from reportlab.platypus import (
        SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle,
        HRFlowable, PageBreak,
    )
    from reportlab.lib.enums import TA_CENTER, TA_LEFT
    REPORTLAB_AVAILABLE = True
except ImportError:
    REPORTLAB_AVAILABLE = False


def _build_styles():
    styles = getSampleStyleSheet()
    title_style = ParagraphStyle(
        "DocTitle", parent=styles["Title"],
        fontSize=22, spaceAfter=6, textColor=colors.HexColor("#1a1a2e"),
    )
    h1 = ParagraphStyle(
        "H1", parent=styles["Heading1"],
        fontSize=14, textColor=colors.HexColor("#16213e"),
        spaceBefore=12, spaceAfter=4,
    )
    h2 = ParagraphStyle(
        "H2", parent=styles["Heading2"],
        fontSize=11, textColor=colors.HexColor("#0f3460"),
        spaceBefore=8, spaceAfter=2,
    )
    body = ParagraphStyle(
        "Body", parent=styles["Normal"],
        fontSize=9, leading=13,
    )
    code_style = ParagraphStyle(
        "Code", parent=styles["Code"],
        fontSize=8, leading=11, backColor=colors.HexColor("#f4f4f4"),
        leftIndent=10, rightIndent=10,
    )
    return title_style, h1, h2, body, code_style


def generate_pdf(
    analysis: dict,
    filename: str,
    output_path: str = "/tmp/doc_genie_report.pdf",
    coverage_before: Optional[float] = None,
) -> str:
    """
    Generate a professional PDF documentation summary.

    Args:
        analysis: Dict from CodeAnalyzer.analyze().
        filename: Original Python filename.
        output_path: Where to write the PDF.
        coverage_before: Optional pre-processing coverage percentage.

    Returns:
        Path to the generated PDF.
    """
    if not REPORTLAB_AVAILABLE:
        raise ImportError("reportlab is not installed. Run: pip install reportlab")

    doc = SimpleDocTemplate(
        output_path,
        pagesize=A4,
        leftMargin=2*cm, rightMargin=2*cm,
        topMargin=2*cm, bottomMargin=2*cm,
    )

    title_style, h1, h2, body, code_style = _build_styles()
    story = []

    # ── Title ──────────────────────────────────────────────────────────────
    story.append(Paragraph("Doc-Genie Professional", title_style))
    story.append(Paragraph("Python Documentation Report", h1))
    story.append(HRFlowable(width="100%", thickness=1, color=colors.HexColor("#0f3460")))
    story.append(Spacer(1, 0.3*cm))

    # ── Meta info ──────────────────────────────────────────────────────────
    now = datetime.now().strftime("%Y-%m-%d %H:%M")
    story.append(Paragraph(f"<b>File:</b> {filename}", body))
    story.append(Paragraph(f"<b>Generated:</b> {now}", body))
    story.append(Spacer(1, 0.4*cm))

    # ── Coverage summary ───────────────────────────────────────────────────
    total = analysis.get("total_functions", 0)
    documented = analysis.get("documented", 0)
    coverage_after = analysis.get("coverage", 0.0)

    story.append(Paragraph("Coverage Summary", h1))

    cov_data = [
        ["Metric", "Value"],
        ["Total Functions", str(total)],
        ["Documented (after)", str(documented)],
        ["Coverage (after)", f"{coverage_after}%"],
    ]
    if coverage_before is not None:
        cov_data.insert(3, ["Coverage (before)", f"{coverage_before}%"])

    cov_table = Table(cov_data, colWidths=[8*cm, 8*cm])
    cov_table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#16213e")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTSIZE", (0, 0), (-1, -1), 9),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f0f4ff")]),
        ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#cccccc")),
        ("ALIGN", (1, 0), (1, -1), "CENTER"),
        ("TOPPADDING", (0, 0), (-1, -1), 5),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 5),
    ]))
    story.append(cov_table)
    story.append(Spacer(1, 0.5*cm))

    # ── Classes ────────────────────────────────────────────────────────────
    classes = analysis.get("classes", [])
    if classes:
        story.append(Paragraph("Classes", h1))
        class_data = [["Class", "Line", "Bases", "Methods", "Has Docstring"]]
        for cls in classes:
            class_data.append([
                cls["name"],
                str(cls["lineno"]),
                ", ".join(cls["bases"]) or "—",
                str(cls["method_count"]),
                "Yes" if cls["existing_docstring"] else "No",
            ])
        t = Table(class_data, colWidths=[4*cm, 1.5*cm, 4*cm, 2*cm, 2.5*cm])
        t.setStyle(TableStyle([
            ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#0f3460")),
            ("TEXTCOLOR", (0, 0), (-1, 0), colors.white),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("FONTSIZE", (0, 0), (-1, -1), 8),
            ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#f0f4ff")]),
            ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#cccccc")),
            ("TOPPADDING", (0, 0), (-1, -1), 4),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 4),
        ]))
        story.append(t)
        story.append(Spacer(1, 0.5*cm))

    # ── Functions ──────────────────────────────────────────────────────────
    functions = analysis.get("functions", [])
    if functions:
        story.append(Paragraph("Function Details", h1))

        for func in functions:
            name = func["name"]
            kind = "async def" if func["is_async"] else "def"
            cls_info = f" (in {func['class_name']})" if func.get("class_name") else ""
            story.append(Paragraph(f"{kind} {name}{cls_info} — Line {func['lineno']}", h2))

            # Parameters table
            params = func.get("params", [])
            if params:
                param_data = [["Parameter", "Type Hint", "Default"]]
                for p in params:
                    param_data.append([
                        p["name"],
                        p.get("annotation") or "—",
                        p.get("default") or "—",
                    ])
                pt = Table(param_data, colWidths=[5*cm, 5*cm, 4*cm])
                pt.setStyle(TableStyle([
                    ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#e8eaf6")),
                    ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
                    ("FONTSIZE", (0, 0), (-1, -1), 8),
                    ("GRID", (0, 0), (-1, -1), 0.5, colors.HexColor("#cccccc")),
                    ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#fafafa")]),
                    ("TOPPADDING", (0, 0), (-1, -1), 3),
                    ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
                ]))
                story.append(pt)

            # Details row
            details = []
            if func.get("return_annotation"):
                details.append(f"<b>Returns:</b> {func['return_annotation']}")
            exc = func.get("raised_exceptions", [])
            if exc:
                details.append(f"<b>Raises:</b> {', '.join(exc)}")
            details.append(f"<b>Complexity:</b> {func.get('complexity_label','—')} ({func.get('complexity',0)})")
            details.append(f"<b>Side Effects:</b> {'Yes' if func.get('has_side_effects') else 'No'}")
            story.append(Paragraph("  |  ".join(details), body))

            logic = func.get("inferred_logic", [])
            if logic:
                story.append(Paragraph("<b>Inferred Logic:</b> " + "; ".join(logic), body))

            if func.get("existing_docstring"):
                snippet = func["existing_docstring"][:150].replace("\n", " ").strip()
                story.append(Paragraph(f"<b>Existing Docstring:</b> {snippet}...", body))

            story.append(Spacer(1, 0.25*cm))
            story.append(HRFlowable(width="100%", thickness=0.5, color=colors.HexColor("#dddddd")))
            story.append(Spacer(1, 0.15*cm))

    doc.build(story)
    return output_path

Writing pdf_exporter.py


In [6]:
%%writefile utils.py
# paste full content here
"""utils.py - Code insertion, indentation helpers, and batch utilities."""
import ast
import re
import os
import textwrap
from typing import Optional


def get_function_indent(source_lines: list[str], lineno: int) -> int:
    """Return the indentation level (in spaces) of the function definition line."""
    line = source_lines[lineno - 1]
    return len(line) - len(line.lstrip())


def insert_docstring_into_source(source_code: str, func_meta: dict, docstring_body: str) -> str:
    """
    Insert or replace a docstring for a function in the source code.

    Args:
        source_code: Full original source as a string.
        func_meta: Metadata dict from CodeAnalyzer.
        docstring_body: The generated docstring (including triple quotes).

    Returns:
        Modified source code string.
    """
    lines = source_code.splitlines(keepends=True)
    lineno = func_meta["lineno"]  # 1-indexed

    # Determine base indentation from the def line
    def_line = lines[lineno - 1]
    base_indent = len(def_line) - len(def_line.lstrip())
    body_indent = base_indent + 4

    # Re-indent docstring body to match function body indent
    doc_lines = docstring_body.splitlines()
    indented_doc_lines = []
    for i, dl in enumerate(doc_lines):
        if i == 0:
            indented_doc_lines.append(" " * body_indent + dl.lstrip())
        else:
            # Preserve relative indentation within docstring
            stripped = dl.lstrip()
            if stripped:
                current_indent = len(dl) - len(stripped)
                indented_doc_lines.append(" " * (body_indent + current_indent) + stripped)
            else:
                indented_doc_lines.append("")
    final_docstring = "\n".join(indented_doc_lines) + "\n"

    # Find the first line of the body (after the def line)
    # Handle multiline def signatures
    body_start = lineno  # 0-indexed: lineno is already past the def
    # Scan for the colon that ends the def
    full_def = ""
    i = lineno - 1
    while i < len(lines):
        full_def += lines[i]
        if ":" in lines[i] and not lines[i].strip().startswith("#"):
            body_start = i + 1
            break
        i += 1

    if func_meta.get("existing_docstring"):
        # Remove existing docstring first
        # Find the docstring start in body
        idx = body_start
        while idx < len(lines) and not lines[idx].strip():
            idx += 1
        # Check if this line starts a docstring
        stripped = lines[idx].strip()
        if stripped.startswith('"""') or stripped.startswith("'''"):
            quote = '"""' if stripped.startswith('"""') else "'''"
            end_idx = idx
            if stripped.count(quote) >= 2 and len(stripped) > 3:
                # single-line docstring
                end_idx = idx
            else:
                end_idx = idx + 1
                while end_idx < len(lines) and quote not in lines[end_idx]:
                    end_idx += 1
            # Remove lines idx..end_idx inclusive
            lines = lines[:idx] + lines[end_idx + 1:]
            body_start = idx

    # Insert new docstring at body_start
    lines.insert(body_start, final_docstring)
    return "".join(lines)


def process_source(source_code: str, analysis: dict, style: str = "google", mode: str = "generate") -> str:
    """
    Apply docstrings to all functions in source_code.

    Args:
        source_code: Original Python source.
        analysis: Dict from CodeAnalyzer.analyze().
        style: Docstring style ('google', 'numpy', 'sphinx').
        mode: 'generate' (overwrite) or 'improve' (merge with existing).

    Returns:
        Modified source code string.
    """
    from generator import generate_docstring, improve_docstring

    functions = analysis.get("functions", [])
    # Sort in reverse order to process bottom-up (preserves line numbers)
    functions = sorted(functions, key=lambda f: f["lineno"], reverse=True)

    modified = source_code
    for meta in functions:
        func_indent = get_function_indent(modified.splitlines(), meta["lineno"])
        body_indent = func_indent + 4

        existing = meta.get("existing_docstring")
        if mode == "improve" and existing:
            docstring = improve_docstring(existing, meta, style, body_indent)
        else:
            docstring = generate_docstring(meta, style, body_indent)

        modified = insert_docstring_into_source(modified, meta, docstring)

    return modified


def save_output(source_code: str, original_path: str, output_dir: str = "/tmp/doc_genie_out") -> str:
    """Save modified source to output directory and return the new filepath."""
    os.makedirs(output_dir, exist_ok=True)
    filename = os.path.basename(original_path)
    out_path = os.path.join(output_dir, f"documented_{filename}")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write(source_code)
    return out_path


def batch_process(file_paths: list[str], style: str = "google", mode: str = "generate") -> list[dict]:
    """
    Process multiple Python files.

    Returns a list of result dicts with keys: filepath, output_path, coverage, error.
    """
    from analyzer import analyze_source

    results = []
    for fp in file_paths:
        try:
            with open(fp, "r", encoding="utf-8") as f:
                source = f.read()
            _, analysis = analyze_source(source)
            if "error" in analysis:
                results.append({"filepath": fp, "error": analysis["error"]})
                continue
            modified = process_source(source, analysis, style, mode)
            out_path = save_output(modified, fp)
            results.append({
                "filepath": fp,
                "output_path": out_path,
                "coverage_before": analysis["coverage"],
                "total_functions": analysis["total_functions"],
                "error": None,
            })
        except Exception as e:
            results.append({"filepath": fp, "error": str(e)})
    return results

Writing utils.py


In [10]:
%%writefile app.py
"""
app.py - Doc-Genie v2.0
Upgraded Gradio interface with dark theme, inline code editor,
share buttons, TXT export, and improved UX.
"""
import os
import tempfile
import traceback
import zipfile
import gradio as gr

from analyzer import analyze_source
from utils import process_source, save_output, batch_process
from pdf_exporter import generate_pdf

OUTPUT_DIR = tempfile.mkdtemp(prefix="doc_genie_")

# ─────────────────────────────────────────────
# Dark theme CSS
# ─────────────────────────────────────────────
DARK_CSS = """
@import url('https://fonts.googleapis.com/css2?family=JetBrains+Mono:wght@400;600&family=Syne:wght@400;600;800&display=swap');

* { box-sizing: border-box; }

body, .gradio-container {
    background: #0d1117 !important;
    color: #e6edf3 !important;
    font-family: 'Syne', sans-serif !important;
}

/* Header */
#header {
    background: linear-gradient(135deg, #0d1117 0%, #161b22 50%, #0d1117 100%);
    border-bottom: 1px solid #30363d;
    padding: 28px 32px 20px;
    margin-bottom: 8px;
}
#header h1 {
    font-family: 'Syne', sans-serif;
    font-size: 2.2em;
    font-weight: 800;
    color: #fff;
    margin: 0 0 4px;
    letter-spacing: -0.5px;
}
#header h1 span { color: #7c3aed; }
#header p {
    color: #8b949e;
    font-size: 0.95em;
    margin: 0 0 2px;
}
#header .badge {
    display: inline-block;
    background: #7c3aed22;
    border: 1px solid #7c3aed55;
    color: #a78bfa;
    font-size: 0.75em;
    padding: 2px 10px;
    border-radius: 20px;
    margin-top: 6px;
    font-family: 'JetBrains Mono', monospace;
}

/* Panels */
.panel-box {
    background: #161b22;
    border: 1px solid #30363d;
    border-radius: 12px;
    padding: 20px;
    margin-bottom: 12px;
}
.panel-title {
    font-size: 0.8em;
    font-weight: 600;
    text-transform: uppercase;
    letter-spacing: 1.5px;
    color: #7c3aed;
    margin-bottom: 12px;
    display: flex;
    align-items: center;
    gap: 6px;
}

/* Buttons */
.gr-button-primary, button.primary {
    background: linear-gradient(135deg, #7c3aed, #5b21b6) !important;
    border: none !important;
    color: white !important;
    font-family: 'Syne', sans-serif !important;
    font-weight: 600 !important;
    border-radius: 8px !important;
    padding: 12px 24px !important;
    transition: all 0.2s ease !important;
}
.gr-button-primary:hover {
    transform: translateY(-1px);
    box-shadow: 0 4px 20px #7c3aed55 !important;
}

/* Tabs */
.tab-nav button {
    background: transparent !important;
    color: #8b949e !important;
    border: none !important;
    border-bottom: 2px solid transparent !important;
    font-family: 'Syne', sans-serif !important;
    font-weight: 600 !important;
    padding: 10px 20px !important;
    transition: all 0.2s !important;
}
.tab-nav button.selected {
    color: #a78bfa !important;
    border-bottom-color: #7c3aed !important;
    background: transparent !important;
}

/* Inputs */
textarea, input[type="text"], .gr-textbox textarea {
    background: #0d1117 !important;
    border: 1px solid #30363d !important;
    color: #e6edf3 !important;
    font-family: 'JetBrains Mono', monospace !important;
    border-radius: 8px !important;
    font-size: 0.88em !important;
}
textarea:focus, input:focus {
    border-color: #7c3aed !important;
    outline: none !important;
    box-shadow: 0 0 0 3px #7c3aed22 !important;
}

/* Code output */
.code-output {
    background: #0d1117;
    border: 1px solid #30363d;
    border-radius: 8px;
    font-family: 'JetBrains Mono', monospace;
    font-size: 0.85em;
}

/* Stat cards */
.stat-card {
    background: #0d1117;
    border: 1px solid #30363d;
    border-radius: 8px;
    padding: 14px 18px;
    text-align: center;
}
.stat-value {
    font-size: 1.8em;
    font-weight: 800;
    color: #a78bfa;
    font-family: 'Syne', sans-serif;
}
.stat-label {
    font-size: 0.75em;
    color: #8b949e;
    text-transform: uppercase;
    letter-spacing: 1px;
}

/* Coverage bar */
.coverage-bar-wrap {
    background: #0d1117;
    border-radius: 4px;
    height: 8px;
    overflow: hidden;
    margin-top: 6px;
}
.coverage-bar-fill {
    height: 100%;
    background: linear-gradient(90deg, #7c3aed, #a78bfa);
    border-radius: 4px;
    transition: width 0.6s ease;
}

/* Dropdown */
.gr-dropdown select, select {
    background: #0d1117 !important;
    border: 1px solid #30363d !important;
    color: #e6edf3 !important;
    border-radius: 8px !important;
}

/* File upload */
.gr-file {
    background: #0d1117 !important;
    border: 1px dashed #30363d !important;
    border-radius: 8px !important;
}

/* Markdown */
.gr-markdown {
    background: transparent !important;
    color: #e6edf3 !important;
}
.gr-markdown h1, .gr-markdown h2, .gr-markdown h3 {
    color: #fff !important;
}
.gr-markdown code {
    background: #0d1117 !important;
    color: #a78bfa !important;
    border: 1px solid #30363d !important;
    border-radius: 4px !important;
    padding: 1px 6px !important;
    font-family: 'JetBrains Mono', monospace !important;
}
.gr-markdown table {
    border-collapse: collapse !important;
    width: 100% !important;
}
.gr-markdown th {
    background: #161b22 !important;
    color: #a78bfa !important;
    padding: 8px 12px !important;
    border: 1px solid #30363d !important;
}
.gr-markdown td {
    padding: 8px 12px !important;
    border: 1px solid #30363d !important;
    color: #e6edf3 !important;
}

/* Share buttons */
.share-wa {
    background: #25D366 !important;
    color: white !important;
    border: none !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
}
.share-fb {
    background: #1877F2 !important;
    color: white !important;
    border: none !important;
    border-radius: 8px !important;
    font-weight: 600 !important;
}

/* Labels */
label, .gr-label {
    color: #8b949e !important;
    font-family: 'Syne', sans-serif !important;
    font-size: 0.85em !important;
    font-weight: 600 !important;
    text-transform: uppercase !important;
    letter-spacing: 0.5px !important;
}

/* Radio buttons */
.gr-radio label { text-transform: none !important; }

/* Scrollbar */
::-webkit-scrollbar { width: 6px; height: 6px; }
::-webkit-scrollbar-track { background: #0d1117; }
::-webkit-scrollbar-thumb { background: #30363d; border-radius: 3px; }
::-webkit-scrollbar-thumb:hover { background: #7c3aed; }
"""


# ─────────────────────────────────────────────
# Processing functions
# ─────────────────────────────────────────────

def process_code_input(code_text, file_obj, style, mode):
    """Process either typed code or uploaded file."""

    # Determine source
    source = None
    filename = "untitled.py"

    if file_obj is not None:
        try:
            with open(file_obj.name, "r", encoding="utf-8") as f:
                source = f.read()
            filename = os.path.basename(file_obj.name)
        except Exception as e:
            return f"❌ Error reading file: {e}", "", "", None, None, None

    elif code_text and code_text.strip():
        source = code_text.strip()
        filename = "pasted_code.py"

    if not source:
        return "❌ Please paste code or upload a .py file.", "", "", None, None, None

    # Analyze
    try:
        _, analysis = analyze_source(source)
    except Exception as e:
        return f"❌ Analysis failed: {e}", "", "", None, None, None

    if "error" in analysis:
        return f"❌ {analysis['error']}", "", "", None, None, None

    coverage_before = analysis["coverage"]

    # Generate docstrings
    try:
        modified = process_source(source, analysis, style.lower(), mode.lower())
    except Exception as e:
        return f"❌ Generation failed: {e}\n{traceback.format_exc()}", "", "", None, None, None

    # Re-analyze for updated coverage
    try:
        _, new_analysis = analyze_source(modified)
        coverage_after = new_analysis["coverage"]
    except Exception:
        coverage_after = 100.0

    # Save .py output
    out_py = save_output(modified, filename, OUTPUT_DIR)

    # Save .txt output
    txt_path = os.path.join(OUTPUT_DIR, f"documented_{os.path.splitext(filename)[0]}.txt")
    with open(txt_path, "w", encoding="utf-8") as f:
        f.write(modified)

    # Generate PDF
    pdf_path = None
    try:
        pdf_path = os.path.join(OUTPUT_DIR, f"doc_report_{os.path.splitext(filename)[0]}.pdf")
        generate_pdf(analysis, filename=filename, output_path=pdf_path, coverage_before=coverage_before)
    except Exception:
        pdf_path = None

    # Build stats markdown
    total = analysis["total_functions"]
    documented_before = analysis["documented"]
    classes = len(analysis.get("classes", []))

    stats_md = f"""
### 📊 Analysis Results

| Metric | Value |
|--------|-------|
| 🔢 Total Functions | **{total}** |
| 📝 Pre-existing Docstrings | **{documented_before}** |
| 📈 Coverage Before | **{coverage_before}%** |
| ✅ Coverage After | **{coverage_after}%** |
| 🏛️ Classes Detected | **{classes}** |
| 🎨 Style | **{style}** |
| ⚙️ Mode | **{mode}** |

**{"🎉 Full coverage achieved!" if coverage_after == 100.0 else f"📈 Improved by {round(coverage_after - coverage_before, 1)}%"}**
"""

    # Status message
    status = f"✅ {style.upper()} Docstrings Generated! {total} functions documented."

    return modified, stats_md, status, out_py, txt_path, pdf_path


def process_batch(file_objs, style, mode):
    """Batch process multiple files."""
    if not file_objs:
        return "❌ No files uploaded.", None

    paths = [f.name for f in file_objs]
    results = batch_process(paths, style.lower(), mode.lower())

    lines = ["## 📦 Batch Processing Results\n"]
    output_files = []

    for r in results:
        name = os.path.basename(r["filepath"])
        if r.get("error"):
            lines.append(f"❌ **{name}** — Error: {r['error']}")
        else:
            lines.append(
                f"✅ **{name}** — {r['total_functions']} functions | "
                f"Coverage: {r['coverage_before']}% → 100%"
            )
            output_files.append(r["output_path"])

    zip_path = None
    if output_files:
        zip_path = os.path.join(OUTPUT_DIR, "batch_documented.zip")
        with zipfile.ZipFile(zip_path, "w") as zf:
            for fp in output_files:
                zf.write(fp, os.path.basename(fp))
        lines.append(f"\n\n📦 **{len(output_files)} files** packed into ZIP")

    return "\n".join(lines), zip_path


def get_share_text(stats_md):
    """Generate share text."""
    return (
        "🧞 Just used Doc-Genie Professional to auto-generate Python docstrings!\n\n"
        "✅ Instant AST analysis\n"
        "✅ Google / NumPy / Sphinx styles\n"
        "✅ 0% → 100% documentation coverage\n\n"
        "Try it yourself! #Python #DocStrings #DocGenie"
    )


# ─────────────────────────────────────────────
# UI
# ─────────────────────────────────────────────

def build_ui():
    with gr.Blocks(title="Doc-Genie v2.0") as demo:

        # Header
        gr.HTML("""
        <div id="header">
            <h1>🧞 Doc-Genie <span>v2.0</span></h1>
            <p>Professional Python Function Docstring Generator</p>
            <p>Transform your Python functions with accurate, AST-powered documentation!</p>
            <span class="badge">⚡ AST Analysis · Google · NumPy · Sphinx</span>
        </div>
        """)

        with gr.Tabs():

            # ── Single File / Code Tab ─────────────────────────────────────
            with gr.TabItem("📄 Generate Docstrings"):
                with gr.Row():

                    # Left column — Input
                    with gr.Column(scale=3):
                        gr.HTML('<div class="panel-title">📝 Input Function Code</div>')

                        code_input = gr.Code(
                            label="Paste Python Code Here",
                            language="python",
                            lines=12,
                            value="""def calculate_factorial(n):
    if n == 0:
        return 1
    return n * calculate_factorial(n - 1)""",
                        )

                        gr.HTML('<div style="text-align:center;color:#8b949e;padding:8px 0;font-size:0.85em;">— or upload a file —</div>')

                        file_input = gr.File(
                            label="Upload .py File",
                            file_types=[".py"],
                        )

                        with gr.Row():
                            style_radio = gr.Radio(
                                choices=["Google", "NumPy", "Sphinx"],
                                value="Google",
                                label="Docstring Style",
                            )
                            mode_radio = gr.Radio(
                                choices=["Generate", "Improve"],
                                value="Generate",
                                label="Mode",
                            )

                        generate_btn = gr.Button(
                            "✨ Generate Docstring",
                            variant="primary",
                            size="lg",
                        )

                        status_box = gr.Textbox(
                            label="Status",
                            interactive=False,
                            lines=1,
                        )

                    # Right column — Controls & Export
                    with gr.Column(scale=1):
                        gr.HTML('<div class="panel-title">🎛️ Controls</div>')
                        clear_btn = gr.Button("🗑️ Clear History", size="sm")

                        gr.HTML('<div class="panel-title" style="margin-top:16px">📤 Export</div>')
                        download_py = gr.File(label="⬇️ Download .py File")
                        download_txt = gr.File(label="⬇️ Download .txt File")
                        download_pdf = gr.File(label="⬇️ Download PDF Report")

                        gr.HTML('<div class="panel-title" style="margin-top:16px">📱 Share</div>')
                        share_text = gr.Textbox(
                            label="Share Text",
                            lines=4,
                            interactive=True,
                            value="Generate docstrings first to get share text...",
                        )
                        gr.HTML("""
                        <a href="https://wa.me/?text=Check%20out%20Doc-Genie%20Professional!"
                           target="_blank"
                           style="display:block;background:#25D366;color:white;text-align:center;
                                  padding:10px;border-radius:8px;font-weight:600;margin-bottom:8px;
                                  text-decoration:none;font-family:'Syne',sans-serif;">
                            📱 Share on WhatsApp
                        </a>
                        <a href="https://www.facebook.com/sharer/sharer.php?u=https://github.com"
                           target="_blank"
                           style="display:block;background:#1877F2;color:white;text-align:center;
                                  padding:10px;border-radius:8px;font-weight:600;
                                  text-decoration:none;font-family:'Syne',sans-serif;">
                            📘 Share on Facebook
                        </a>
                        """)

                # Output section
                gr.HTML('<div class="panel-title" style="margin-top:8px">📋 Generated Output</div>')
                with gr.Row():
                    with gr.Column(scale=3):
                        code_output = gr.Code(
                            label="Function with Docstring",
                            language="python",
                            lines=18,
                            interactive=False,
                        )
                    with gr.Column(scale=1):
                        stats_output = gr.Markdown(label="Analysis Summary")

            # ── Batch Tab ─────────────────────────────────────────────────
            with gr.TabItem("📦 Batch Processing"):
                with gr.Row():
                    with gr.Column(scale=1):
                        batch_files = gr.File(
                            label="Upload Multiple .py Files",
                            file_types=[".py"],
                            file_count="multiple",
                        )
                        batch_style = gr.Dropdown(
                            choices=["Google", "NumPy", "Sphinx"],
                            value="Google",
                            label="Style",
                        )
                        batch_mode = gr.Dropdown(
                            choices=["Generate", "Improve"],
                            value="Generate",
                            label="Mode",
                        )
                        batch_btn = gr.Button("🚀 Process All Files", variant="primary")

                    with gr.Column(scale=2):
                        batch_result = gr.Markdown(label="Results")
                        batch_download = gr.File(label="⬇️ Download ZIP")

                batch_btn.click(
                    fn=process_batch,
                    inputs=[batch_files, batch_style, batch_mode],
                    outputs=[batch_result, batch_download],
                )

            # ── Help Tab ──────────────────────────────────────────────────
            with gr.TabItem("ℹ️ Help"):
                help_text = (
                    "## Doc-Genie v2.0 — User Guide\n\n"
                    "### Features\n"
                    "- **AST Analysis**: Extracts params, type hints, decorators, return types, exceptions\n"
                    "- **Logic Inference**: Detects math, DB, file I/O, API calls, validation, iteration\n"
                    "- **Complexity**: Cyclomatic complexity (Low / Medium / High)\n"
                    "- **3 Styles**: Google, NumPy, Sphinx\n"
                    "- **Improve Mode**: Merges with existing docstrings\n"
                    "- **Batch Mode**: Process many files, download as ZIP\n"
                    "- **PDF + TXT Export**: Download documentation in multiple formats\n\n"
                    "### How to Use\n"
                    "1. Paste Python code directly OR upload a .py file\n"
                    "2. Choose a documentation style (Google / NumPy / Sphinx)\n"
                    "3. Choose mode: Generate (new) or Improve (merge with existing)\n"
                    "4. Click Generate Docstring\n"
                    "5. Download .py, .txt, or PDF report\n\n"
                    "### Docstring Styles\n"
                    "| Style | Best For |\n"
                    "|-------|----------|\n"
                    "| Google | General Python projects |\n"
                    "| NumPy | Scientific / data projects |\n"
                    "| Sphinx | Auto-documentation websites |\n\n"
                    "### Tips\n"
                    "- Use **Improve** mode to keep existing docstrings and only fill in missing sections\n"
                    "- Use **Batch** tab to document an entire project at once\n"
                    "- The PDF report is great for sharing documentation with your team\n"
                )
                gr.Markdown(help_text)

        # ── Event handlers ────────────────────────────────────────────────
        generate_btn.click(
            fn=process_code_input,
            inputs=[code_input, file_input, style_radio, mode_radio],
            outputs=[code_output, stats_output, status_box, download_py, download_txt, download_pdf],
        )

        def update_share(stats):
            if not stats or "Results" not in stats:
                return "Generate docstrings first..."
            return (
                "I just used Doc-Genie v2.0 to auto-document my Python code!\n\n"
                "AST-powered docstring generation:\n"
                "✅ Google / NumPy / Sphinx styles\n"
                "✅ 0% to 100% coverage instantly\n"
                "✅ PDF report included\n\n"
                "#Python #Documentation #DocGenie"
            )

        generate_btn.click(fn=update_share, inputs=[stats_output], outputs=[share_text])

        def clear_all():
            return "", None, "", "", None, None, None

        clear_btn.click(
            fn=clear_all,
            outputs=[code_input, file_input, code_output, status_box, download_py, download_txt, download_pdf],
        )

    return demo


if __name__ == "__main__":
    demo = build_ui()
    demo.launch(server_name="0.0.0.0", share=True)

Overwriting app.py


In [ ]:
!python app.py


* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://34b366707e8c1ff9ec.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
